In [8]:
import os
import gc
import math
import random
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import TensorDataset, DataLoader, RandomSampler, SequentialSampler
from transformers import AutoTokenizer, AutoModel, AutoConfig
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, f1_score

print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())


torch: 2.10.0+cu128
CUDA available: True


In [9]:
SEED = 42  


MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512          
NUM_CLASSES = 6
DROPOUT = 0.30           


BATCH_SIZE = 8             
LEARNING_RATE = 1e-5       
WEIGHT_DECAY = 0.01       
FOCAL_GAMMA = 2.0          
MAX_EPOCHS = 30            
EARLY_STOP_PATIENCE = 10   
GRAD_CLIP_NORM = 1.0       


NUM_TEST_PROJECTS_PER_FOLD = 25   
MIN_TEST_EXAMPLES_PER_CLASS = 4   
VALIDATION_FRACTION = 0.20        


USE_SMOTE_OVERSAMPLING = False
SMOTE_TARGET_PER_FLAKY_CLASS = 500   

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


Device: cuda


In [10]:
def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed()
print("Seed set to", SEED)


Seed set to 42


## Dataset

In [11]:
DATA_PATH = "/kaggle/input/datasets/zaimast7454/flakylens-dataset/FlakyLens_dataset_with_nonflaky_indented.csv"  

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())


Shape: (8574, 6)


,id,project,test_name,full_code,label,category
0,1,apache_hadoop,TestDelegationTokenRenewer.testAddRemoveRenewA...,@Test\npublic void testAddRemoveRenewAction() ...,async wait,0
1,2,neo4j_neo4j,RobustJobSchedulerWrapperTest.shouldBeAbleToCa...,@Test\npublic void shouldBeAbleToCancelJob() t...,concurrency,1
2,3,eclipse_xtext-core,RequestManagerTest.testRunWriteAfterRead,@Test\npublic void testRunWriteAfterRead() {\n...,concurrency,1
3,4,NationalSecurityAgency_timely,MetricAdapterTest.testToMetricResponse,@Test\npublic void testToMetricResponse() thro...,unordered collections,3
4,5,apache_hadoop,TestPathData.testToFile,@Test\npublic void testToFile() throws Excepti...,test order dependency,4


In [12]:
LABEL_NAMES = {
    0: "Async Wait",
    1: "Concurrency",
    2: "Time",
    3: "Unordered Collections",
    4: "Test Order Dependency",
    5: "Non-flaky",
}

def normalize_dataset(df):
    df = df.copy()
    required = {"project", "full_code", "category"}
    if not required.issubset(df.columns):
        raise ValueError(f"Expected columns {required}, found {df.columns.tolist()}")

    df = df[["project", "full_code", "category"]].copy()
    df["project"] = df["project"].astype(str)
    df["full_code"] = df["full_code"].fillna("").astype(str)
    df["category"] = pd.to_numeric(df["category"], errors="raise").astype(int)

    if not set(df["category"].unique()).issubset(set(range(NUM_CLASSES))):
        raise ValueError("category values must be integers 0-5")

    return df.reset_index(drop=True)

data = normalize_dataset(df)
print("Number of projects:", data["project"].nunique())
print("\nClass distribution:")
display(data["category"].value_counts().sort_index().rename(index=LABEL_NAMES).to_frame("count"))
print("\nFlaky %:", round(100 * (data["category"] != 5).mean(), 2))


Number of projects: 98

Class distribution:


,count
category,
Async Wait,76
Concurrency,37
Time,33
Unordered Collections,41
Test Order Dependency,93
Non-flaky,8294



Flaky %: 3.27


In [27]:
def create_train_test_groups(df, num_test_projects=NUM_TEST_PROJECTS_PER_FOLD,
                              min_examples_per_class=MIN_TEST_EXAMPLES_PER_CLASS, seed=SEED):
    rng = random.Random(seed)
    unique_projects = df["project"].drop_duplicates().tolist()
    rng.shuffle(unique_projects)

    num_groups = math.ceil(len(unique_projects) / num_test_projects)
    groups = []
    used_projects = set()

    for _ in range(num_groups - 1):
        available_projects = [p for p in unique_projects if p not in used_projects]
        test_projects = available_projects[:num_test_projects]
        test_dataset = df[df["project"].isin(test_projects)]

        
        for _attempt in range(2000):
            category_counts = test_dataset["category"].value_counts()
            missing = [c for c in range(NUM_CLASSES) if category_counts.get(c, 0) < min_examples_per_class]
            if not missing:
                break
            rng.shuffle(available_projects)
            test_projects = available_projects[:num_test_projects]
            test_dataset = df[df["project"].isin(test_projects)]
        else:
            raise RuntimeError("Could not satisfy the minimum-per-class test requirement.")

        used_projects.update(test_projects)
        train_dataset = df[~df["project"].isin(test_projects)]
        groups.append((train_dataset.copy(), test_dataset.copy()))


    remaining = [p for p in unique_projects if p not in used_projects]
    if remaining:
        test_dataset = df[df["project"].isin(remaining)]
        train_dataset = df[~df["project"].isin(remaining)]
        groups.append((train_dataset.copy(), test_dataset.copy()))

    return groups

fold_groups = create_train_test_groups(data)
print(f"Created {len(fold_groups)} fold(s).\n")
for i, (train_pool, test_df) in enumerate(fold_groups, 1):
    overlap = set(train_pool["project"]) & set(test_df["project"])
    assert not overlap, f"Fold {i}: project leakage detected!"
    print(f"Fold {i}: {test_df['project'].nunique()} test projects | "
          f"train rows={len(train_pool)} | test rows={len(test_df)} | "
          f"test class counts={test_df['category'].value_counts().sort_index().to_dict()}")


Created 4 fold(s).

Fold 1: 25 test projects | train rows=6543 | test rows=2031 | test class counts={0: 12, 1: 15, 2: 9, 3: 11, 4: 5, 5: 1979}
Fold 2: 25 test projects | train rows=6362 | test rows=2212 | test class counts={0: 24, 1: 9, 2: 5, 3: 8, 4: 79, 5: 2087}
Fold 3: 25 test projects | train rows=6272 | test rows=2302 | test class counts={0: 21, 1: 4, 2: 12, 3: 8, 4: 6, 5: 2251}
Fold 4: 23 test projects | train rows=6545 | test rows=2029 | test class counts={0: 19, 1: 9, 2: 7, 3: 14, 4: 3, 5: 1977}


## CodeBERT architecture

In [14]:
class BERT_Arch(nn.Module):
    def __init__(self, auto_model, num_classes=NUM_CLASSES):
        super().__init__()
        self.bert = auto_model
        self.dropout = nn.Dropout(DROPOUT)
        self.relu = nn.ReLU()
        self.fc1 = nn.Linear(768, 512)
        self.fc2 = nn.Linear(512, num_classes)
        self.log_softmax = nn.LogSoftmax(dim=-1)

    def forward(self, input_ids, attention_mask):
        input_ids = input_ids.long()
        attention_mask = attention_mask.long()
        outputs = self.bert(input_ids, attention_mask=attention_mask)
        pooled = outputs[1]  

        x = self.fc1(pooled)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return self.log_softmax(x)


def build_fresh_model():

    model_config = AutoConfig.from_pretrained(MODEL_NAME, return_dict=False, output_hidden_states=True)
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    auto_model = AutoModel.from_pretrained(MODEL_NAME, config=model_config)
    model = BERT_Arch(auto_model, NUM_CLASSES).to(DEVICE)
    return model, tokenizer


In [15]:
USE_RAW_LOGITS_LOSS = False  

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0, reduction="mean"):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, model_output, targets):
        
        ce_loss = F.cross_entropy(model_output, targets, reduction="none", weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = (1 - pt) ** self.gamma * ce_loss
        if self.reduction == "mean":
            return focal_loss.mean()
        if self.reduction == "sum":
            return focal_loss.sum()
        return focal_loss


def compute_class_balanced_focal_loss(train_labels):
    class_weights_np = compute_class_weight(
        class_weight="balanced",
        classes=np.arange(NUM_CLASSES),
        y=train_labels,
    )
    weights = torch.tensor(class_weights_np, dtype=torch.float32, device=DEVICE)
    return FocalLoss(alpha=weights, gamma=FOCAL_GAMMA), weights


In [16]:
def apply_smote_like_released_code(train_df):
  
    from imblearn.over_sampling import SMOTE
    from sklearn.feature_extraction.text import TfidfVectorizer

    x_text = train_df["full_code"].reset_index(drop=True)
    y = train_df["category"].reset_index(drop=True)

    vectorizer = TfidfVectorizer(max_features=5000)
    x_vec = vectorizer.fit_transform(x_text)

    sampling_strategy = {c: SMOTE_TARGET_PER_FLAKY_CLASS for c in range(5)}  
    counts = y.value_counts().to_dict()
    sampling_strategy = {c: n for c, n in sampling_strategy.items() if counts.get(c, 0) < n}

    if not sampling_strategy:
        return train_df  

    smote = SMOTE(sampling_strategy=sampling_strategy, random_state=SEED)
    x_res, y_res = smote.fit_resample(x_vec, y)

    num_synthetic = x_res.shape[0] - len(x_text)
    resampled_text = list(x_text) + ["[SMOTE-generated-sample]"] * num_synthetic

    return pd.DataFrame({"full_code": resampled_text, "category": y_res})


In [17]:
def tokenize_texts(tokenizer, texts):
    return tokenizer(
        texts.tolist(),
        max_length=MAX_LENGTH,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )

def make_loader(tokens, labels, batch_size, shuffle):
    dataset = TensorDataset(tokens["input_ids"], tokens["attention_mask"],
                             torch.tensor(labels, dtype=torch.long))
    sampler = RandomSampler(dataset) if shuffle else SequentialSampler(dataset)
    return DataLoader(dataset, sampler=sampler, batch_size=batch_size)


## Train 

In [18]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)  
        optimizer.step()
        total_loss += loss.item()
    return total_loss / max(1, len(loader))


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    preds_all, labels_all = [], []
    for input_ids, attention_mask, labels in loader:
        input_ids, attention_mask, labels = input_ids.to(device), attention_mask.to(device), labels.to(device)
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        total_loss += loss.item()
        preds_all.append(torch.argmax(outputs, dim=1).cpu().numpy())
        labels_all.append(labels.cpu().numpy())
    avg_loss = total_loss / max(1, len(loader))
    return avg_loss, np.concatenate(preds_all), np.concatenate(labels_all)


In [19]:
class EarlyStopping:
    def __init__(self, patience=EARLY_STOP_PATIENCE, delta=0.0):
        self.patience = patience
        self.delta = delta
        self.counter = 0
        self.best_score = None
        self.best_state = None
        self.early_stop = False

    def __call__(self, valid_f1, model):
        score = valid_f1
        if self.best_score is None or score > self.best_score + self.delta:
            self.best_score = score
            self.best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True


## Folds



In [20]:
def train_one_fold(fold_idx, train_pool, test_df, output_dir):
    print(f"\n{'='*70}\nFOLD {fold_idx}\n{'='*70}")

  
    model, tokenizer = build_fresh_model()

    
    train_df, valid_df = train_test_split(
        train_pool, test_size=VALIDATION_FRACTION, random_state=SEED,
        stratify=train_pool["category"],
    )
    train_df = train_df.reset_index(drop=True)
    valid_df = valid_df.reset_index(drop=True)

    assert not (set(train_df["project"]) & set(test_df["project"]))
    assert not (set(valid_df["project"]) & set(test_df["project"]))

    if USE_SMOTE_OVERSAMPLING:
        train_df = apply_smote_like_released_code(train_df)

    print(f"train={len(train_df)}  valid={len(valid_df)}  test={len(test_df)}")

  
    tokens_train = tokenize_texts(tokenizer, train_df["full_code"])
    tokens_valid = tokenize_texts(tokenizer, valid_df["full_code"])
    tokens_test = tokenize_texts(tokenizer, test_df["full_code"])

    train_loader = make_loader(tokens_train, train_df["category"].to_numpy(), BATCH_SIZE, shuffle=True)
    valid_loader = make_loader(tokens_valid, valid_df["category"].to_numpy(), BATCH_SIZE, shuffle=False)
    test_loader  = make_loader(tokens_test,  test_df["category"].to_numpy(),  BATCH_SIZE, shuffle=False)

    
    criterion, class_weights = compute_class_balanced_focal_loss(train_df["category"].to_numpy())
    print("Class weights:", {LABEL_NAMES[i]: round(w, 3) for i, w in enumerate(class_weights.cpu().numpy())})

    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    early_stopping = EarlyStopping(patience=EARLY_STOP_PATIENCE)

    history = []
    for epoch in range(1, MAX_EPOCHS + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, DEVICE)
        valid_loss, valid_preds, valid_labels = evaluate(model, valid_loader, criterion, DEVICE)
        valid_f1 = f1_score(valid_labels, valid_preds, average="macro", zero_division=0)

        history.append({"epoch": epoch, "train_loss": train_loss, "valid_loss": valid_loss, "valid_f1": valid_f1})
        print(f"  epoch {epoch:02d} | train_loss={train_loss:.4f} | valid_loss={valid_loss:.4f} | valid_macroF1={valid_f1:.4f}")

        early_stopping(valid_f1, model)
        if early_stopping.early_stop:
            print(f"  Early stopping at epoch {epoch} (best valid macro-F1={early_stopping.best_score:.4f})")
            break


    if early_stopping.best_state is not None:
        model.load_state_dict(early_stopping.best_state)

   
    test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
    report = classification_report(
        test_labels, test_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True, zero_division=0,
    )
    print("\nFold", fold_idx, "test macro-F1:", report["macro avg"]["f1-score"])

   
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    torch.save(model.state_dict(), output_dir / f"flakylens_fold{fold_idx}.pt")

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "fold": fold_idx,
        "history": pd.DataFrame(history),
        "report": report,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "confusion_matrix": confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))),
    }


## Fold 1

In [28]:
OUTPUT_DIR = "/kaggle/working/flakylens_models"

fold_results = []
for i, (train_pool, test_df) in enumerate(fold_groups[:1], 1):
    result = train_one_fold(i, train_pool, test_df, OUTPUT_DIR)
    fold_results.append(result)


FOLD 1


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

train=5234  valid=1309  test=2031
Class weights: {'Async Wait': np.float32(17.105), 'Concurrency': np.float32(48.463), 'Time': np.float32(45.912), 'Unordered Collections': np.float32(36.347), 'Test Order Dependency': np.float32(12.462), 'Non-flaky': np.float32(0.173)}
  epoch 01 | train_loss=1.5743 | valid_loss=1.4623 | valid_macroF1=0.3107
  epoch 02 | train_loss=1.4626 | valid_loss=1.3651 | valid_macroF1=0.3703
  epoch 03 | train_loss=1.3889 | valid_loss=1.1760 | valid_macroF1=0.4929
  epoch 04 | train_loss=1.0206 | valid_loss=1.1743 | valid_macroF1=0.5921
  epoch 05 | train_loss=1.0494 | valid_loss=0.8558 | valid_macroF1=0.6589
  epoch 06 | train_loss=0.8948 | valid_loss=1.4779 | valid_macroF1=0.6026
  epoch 07 | train_loss=0.7476 | valid_loss=1.6069 | valid_macroF1=0.5724
  epoch 08 | train_loss=0.7749 | valid_loss=1.7121 | valid_macroF1=0.6375
  epoch 09 | train_loss=0.6242 | valid_loss=1.5993 | valid_macroF1=0.5872
  epoch 10 | train_loss=0.5640 | valid_loss=1.3797 | valid_macroF

In [23]:
def load_fold_model_from_checkpoint(checkpoint_path):
    model, tokenizer = build_fresh_model()
    state_dict = torch.load(checkpoint_path, map_location=DEVICE)
    missing, unexpected = model.load_state_dict(state_dict, strict=True)
    model.eval()
    return model, tokenizer


def evaluate_fold_from_checkpoint(fold_idx, checkpoint_path, train_df, test_df):
    print(f"\n{'='*70}\nFOLD {fold_idx} -- evaluating checkpoint (no training)\n{'='*70}")
    print("Checkpoint:", checkpoint_path)

    model, tokenizer = load_fold_model_from_checkpoint(checkpoint_path)

    tokens_test = tokenize_texts(tokenizer, test_df["full_code"])
    test_loader = make_loader(tokens_test, test_df["category"].to_numpy(), BATCH_SIZE, shuffle=False)

    criterion, _ = compute_class_balanced_focal_loss(train_df["category"].to_numpy())

    test_loss, test_preds, test_labels = evaluate(model, test_loader, criterion, DEVICE)
    report = classification_report(
        test_labels, test_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True, zero_division=0,
    )
    print("Fold", fold_idx, "test macro-F1:", report["macro avg"]["f1-score"])

    del model
    gc.collect()
    torch.cuda.empty_cache()

    return {
        "fold": fold_idx,
        "history": None,
        "report": report,
        "test_preds": test_preds,
        "test_labels": test_labels,
        "confusion_matrix": confusion_matrix(test_labels, test_preds, labels=list(range(NUM_CLASSES))),
    }

In [24]:
import pickle

In [24]:
train_pool, test_df = fold_groups[0]
result = evaluate_fold_from_checkpoint(1,"/kaggle/input/datasets/zaimast7454/fourfolds/flakylens_fold1.pt", train_pool, test_df)



FOLD 1 -- evaluating checkpoint (no training)
Checkpoint: /kaggle/input/datasets/zaimast7454/fourfolds/flakylens_fold1.pt


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Fold 1 test macro-F1: 0.6397727272727273


In [1]:
from pathlib import Path
import pickle

Path("/kaggle/working/flakylens_models").mkdir(parents=True, exist_ok=True)

with open("/kaggle/input/datasets/zaimast7454/fourfoldw/fold1_result.pkl", "wb") as f:
    pickle.dump(result, f)

print("Saved.")

The history saving thread hit an unexpected error (OperationalError('attempt to write a readonly database')).History will not be written to the database.


OSError: [Errno 30] Read-only file system: '/kaggle/input/datasets/zaimast7454/fourfoldw/fold1_result.pkl'

## Fold 2

In [15]:
FOLD_TO_RUN = 2  

import pickle

OUTPUT_DIR = "/kaggle/working/flakylens_models"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_pool, test_df = fold_groups[FOLD_TO_RUN - 1]
result = train_one_fold(FOLD_TO_RUN, train_pool, test_df, OUTPUT_DIR)


with open(Path(OUTPUT_DIR) / f"fold{FOLD_TO_RUN}_result.pkl", "wb") as f:
    pickle.dump(result, f)

print(f"Fold {FOLD_TO_RUN} done. Test macro-F1: {result['report']['macro avg']['f1-score']:.4f}")


FOLD 2


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

train=5089  valid=1273  test=2212
Class weights: {'Async Wait': np.float32(20.194), 'Concurrency': np.float32(38.553), 'Time': np.float32(36.877), 'Unordered Collections': np.float32(32.622), 'Test Order Dependency': np.float32(77.106), 'Non-flaky': np.float32(0.171)}
  epoch 01 | train_loss=1.5820 | valid_loss=1.5250 | valid_macroF1=0.2422
  epoch 02 | train_loss=1.4901 | valid_loss=1.4444 | valid_macroF1=0.2521
  epoch 03 | train_loss=1.3293 | valid_loss=1.2347 | valid_macroF1=0.3688
  epoch 04 | train_loss=1.1830 | valid_loss=1.0581 | valid_macroF1=0.5068
  epoch 05 | train_loss=0.8763 | valid_loss=1.2214 | valid_macroF1=0.6551
  epoch 06 | train_loss=0.5784 | valid_loss=1.1086 | valid_macroF1=0.6609
  epoch 07 | train_loss=0.3657 | valid_loss=0.9701 | valid_macroF1=0.8792
  epoch 08 | train_loss=0.1376 | valid_loss=1.0861 | valid_macroF1=0.7764
  epoch 09 | train_loss=0.0518 | valid_loss=0.9762 | valid_macroF1=0.8665
  epoch 10 | train_loss=0.0739 | valid_loss=1.3318 | valid_macroF

## Fold 3

In [25]:
FOLD_TO_RUN = 3   

import pickle

OUTPUT_DIR = "/kaggle/working/flakylens_models"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_pool, test_df = fold_groups[FOLD_TO_RUN - 1]
result = train_one_fold(FOLD_TO_RUN, train_pool, test_df, OUTPUT_DIR)




FOLD 3


config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

train=5017  valid=1255  test=2302
Class weights: {'Async Wait': np.float32(19.004), 'Concurrency': np.float32(32.16), 'Time': np.float32(49.186), 'Unordered Collections': np.float32(32.16), 'Test Order Dependency': np.float32(11.945), 'Non-flaky': np.float32(0.173)}
  epoch 01 | train_loss=1.5548 | valid_loss=1.5967 | valid_macroF1=0.2512
  epoch 02 | train_loss=1.5526 | valid_loss=1.5089 | valid_macroF1=0.3349
  epoch 03 | train_loss=1.3212 | valid_loss=1.2876 | valid_macroF1=0.3829
  epoch 04 | train_loss=0.9912 | valid_loss=0.9543 | valid_macroF1=0.4883
  epoch 05 | train_loss=0.7121 | valid_loss=0.8201 | valid_macroF1=0.7346
  epoch 06 | train_loss=0.5845 | valid_loss=0.9559 | valid_macroF1=0.6584
  epoch 07 | train_loss=0.5010 | valid_loss=1.3057 | valid_macroF1=0.6699
  epoch 08 | train_loss=0.3136 | valid_loss=1.3598 | valid_macroF1=0.7139
  epoch 09 | train_loss=0.2149 | valid_loss=1.1631 | valid_macroF1=0.7967
  epoch 10 | train_loss=0.0363 | valid_loss=1.8446 | valid_macroF1=

In [26]:
with open(Path(OUTPUT_DIR) / f"fold{FOLD_TO_RUN}_result.pkl", "wb") as f:
    pickle.dump(result, f)

print(f"Fold {FOLD_TO_RUN} done. Test macro-F1: {result['report']['macro avg']['f1-score']:.4f}")

Fold 3 done. Test macro-F1: 0.6747


## Fold 4

In [28]:
FOLD_TO_RUN = 4   

import pickle

OUTPUT_DIR = "/kaggle/working/flakylens_models"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

train_pool, test_df = fold_groups[FOLD_TO_RUN - 1]
result = train_one_fold(FOLD_TO_RUN, train_pool, test_df, OUTPUT_DIR)

with open(Path(OUTPUT_DIR) / f"fold{FOLD_TO_RUN}_result.pkl", "wb") as f:
    pickle.dump(result, f)

print(f"Fold {FOLD_TO_RUN} done. Test macro-F1: {result['report']['macro avg']['f1-score']:.4f}")


FOLD 4


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

train=5236  valid=1309  test=2029
Class weights: {'Async Wait': np.float32(18.971), 'Concurrency': np.float32(39.667), 'Time': np.float32(41.556), 'Unordered Collections': np.float32(39.667), 'Test Order Dependency': np.float32(12.12), 'Non-flaky': np.float32(0.173)}
  epoch 01 | train_loss=1.5571 | valid_loss=1.4130 | valid_macroF1=0.2619
  epoch 02 | train_loss=1.4419 | valid_loss=1.3613 | valid_macroF1=0.3657
  epoch 03 | train_loss=1.2378 | valid_loss=1.2398 | valid_macroF1=0.3816
  epoch 04 | train_loss=0.9501 | valid_loss=0.9616 | valid_macroF1=0.5742
  epoch 05 | train_loss=0.6580 | valid_loss=0.8387 | valid_macroF1=0.7695
  epoch 06 | train_loss=0.5092 | valid_loss=0.9638 | valid_macroF1=0.7639
  epoch 07 | train_loss=0.3384 | valid_loss=1.1338 | valid_macroF1=0.7449
  epoch 08 | train_loss=0.1524 | valid_loss=1.1852 | valid_macroF1=0.7751
  epoch 09 | train_loss=0.1218 | valid_loss=1.4493 | valid_macroF1=0.7368
  epoch 10 | train_loss=0.0352 | valid_loss=1.1953 | valid_macroF1

## Aggregate fold results 

In [2]:
import pickle
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report

INPUT_DIR = Path("/kaggle/input/datasets/zaimast7454/fourfoldw")  

NUM_CLASSES = 6
LABEL_NAMES = {
    0: "Async Wait", 1: "Concurrency", 2: "Time",
    3: "Unordered Collections", 4: "Test Order Dependency", 5: "Non-flaky",
}

fold_results = []
for i in range(1, 5):
    with open(INPUT_DIR / f"fold{i}_result.pkl", "rb") as f:
        r = pickle.load(f)
    fold_results.append(r)
    print(f"Fold {i}: test macro-F1 = {r['report']['macro avg']['f1-score']:.4f}")

print(f"\n{len(fold_results)}/4 folds loaded.")

Fold 1: test macro-F1 = 0.6398
Fold 2: test macro-F1 = 0.4763
Fold 3: test macro-F1 = 0.6747
Fold 4: test macro-F1 = 0.6055

4/4 folds loaded.


In [3]:
def summarize_folds(fold_results):
    rows = []
    for r in fold_results:
        row = {"fold": r["fold"]}
        for i in range(NUM_CLASSES):
            row[LABEL_NAMES[i]] = r["report"][LABEL_NAMES[i]]["f1-score"] * 100
        row["Macro Avg"] = r["report"]["macro avg"]["f1-score"] * 100
        rows.append(row)
    per_fold = pd.DataFrame(rows).set_index("fold")


    all_preds = np.concatenate([r["test_preds"] for r in fold_results])
    all_labels = np.concatenate([r["test_labels"] for r in fold_results])
    pooled_report = classification_report(
        all_labels, all_preds, labels=list(range(NUM_CLASSES)),
        target_names=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
        output_dict=True, zero_division=0,
    )
    overall = {"fold": "Overall (pooled)"}
    for i in range(NUM_CLASSES):
        overall[LABEL_NAMES[i]] = pooled_report[LABEL_NAMES[i]]["f1-score"] * 100
    overall["Macro Avg"] = pooled_report["macro avg"]["f1-score"] * 100

    return per_fold, pd.DataFrame([overall]).set_index("fold"), pooled_report

per_fold_table, overall_table, pooled_report = summarize_folds(fold_results)

print("Per-fold F1-score (%):")
display(per_fold_table.round(2))

print("\nPooled across all folds -- comparable to paper Table 2's 'FlakyLens (Ours)' row:")
display(overall_table.round(2))

Per-fold F1-score (%):


,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky,Macro Avg
fold,,,,,,,
1,60.00,0.00,87.50,81.82,54.55,100.00,63.98
2,45.57,26.67,53.33,30.43,29.79,99.98,47.63
3,68.18,40.00,80.00,66.67,50.00,99.98,67.47
4,69.77,20.00,73.68,66.67,33.33,99.82,60.55



Pooled across all folds -- comparable to paper Table 2's 'FlakyLens (Ours)' row:


,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky,Macro Avg
fold,,,,,,,
Overall (pooled),58.25,20.0,74.29,54.55,34.11,99.95,56.86


In [4]:
cm_total = sum(r["confusion_matrix"] for r in fold_results)
cm_df = pd.DataFrame(cm_total,
                      index=[LABEL_NAMES[i] for i in range(NUM_CLASSES)],
                      columns=[LABEL_NAMES[i] for i in range(NUM_CLASSES)])
print("Confusion matrix (rows=actual, cols=predicted):")
display(cm_df)

Confusion matrix (rows=actual, cols=predicted):


,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky
Async Wait,60,6,4,1,4,1
Concurrency,30,5,1,0,1,0
Time,6,1,26,0,0,0
Unordered Collections,5,0,1,30,5,0
Test Order Dependency,29,1,5,35,22,1
Non-flaky,0,0,0,3,4,8287


In [5]:
paper_table2 = pd.DataFrame({
    "Async Wait": [42.80, 43.75, 58.37],
    "Concurrency": [29.62, 20.00, 35.92],
    "Time": [72.55, 53.75, 72.73],
    "Unordered Collections": [39.15, 58.75, 73.63],
    "Test Order Dependency": [54.70, 36.25, 64.35],
    "Non-flaky": [100.00, 99.50, 100.00],
    "Macro Avg": [56.62, 52.00, 65.79],
}, index=["Flakify (paper)", "FlakyCat (paper)", "FlakyLens (paper)"])

comparison = pd.concat([paper_table2, overall_table.rename(index={"Overall (pooled)": "FlakyLens (yours)"})])
display(comparison.round(2))

,Async Wait,Concurrency,Time,Unordered Collections,Test Order Dependency,Non-flaky,Macro Avg
Flakify (paper),42.80,29.62,72.55,39.15,54.70,100.00,56.62
FlakyCat (paper),43.75,20.00,53.75,58.75,36.25,99.50,52.00
FlakyLens (paper),58.37,35.92,72.73,73.63,64.35,100.00,65.79
FlakyLens (yours),58.25,20.00,74.29,54.55,34.11,99.95,56.86


In [6]:
SAVE_DIR = Path("/kaggle/working/flakylens_models")
SAVE_DIR.mkdir(parents=True, exist_ok=True)
per_fold_table.to_csv(SAVE_DIR / "per_fold_f1.csv")
overall_table.to_csv(SAVE_DIR / "overall_f1.csv")
cm_df.to_csv(SAVE_DIR / "confusion_matrix.csv")
comparison.to_csv(SAVE_DIR / "comparison_vs_paper.csv")
print("Saved to", SAVE_DIR)

Saved to /kaggle/working/flakylens_models


In [23]:
for i, (train_pool, test_df) in enumerate(fold_groups, 1):
    counts = test_df["category"].value_counts().reindex(range(6), fill_value=0)
    print(f"Fold {i} test set category counts: {dict(zip(LABEL_NAMES.values(), counts))}")

Fold 1 test set category counts: {'Async Wait': 12, 'Concurrency': 15, 'Time': 9, 'Unordered Collections': 11, 'Test Order Dependency': 5, 'Non-flaky': 1979}
Fold 2 test set category counts: {'Async Wait': 24, 'Concurrency': 9, 'Time': 5, 'Unordered Collections': 8, 'Test Order Dependency': 79, 'Non-flaky': 2087}
Fold 3 test set category counts: {'Async Wait': 21, 'Concurrency': 4, 'Time': 12, 'Unordered Collections': 8, 'Test Order Dependency': 6, 'Non-flaky': 2251}
Fold 4 test set category counts: {'Async Wait': 19, 'Concurrency': 9, 'Time': 7, 'Unordered Collections': 14, 'Test Order Dependency': 3, 'Non-flaky': 1977}


In [22]:
for i, (train_pool, test_df) in enumerate(fold_groups, 1):
    print(f"Fold {i}: category 3 examples =", test_df[test_df.category==3]["full_code"].iloc[0][:80] if (test_df.category==3).any() else "none")
    print(f"Fold {i}: category 4 examples =", test_df[test_df.category==4]["full_code"].iloc[0][:80] if (test_df.category==4).any() else "none")

Fold 1: category 3 examples = @Test
public void test_multimap() throws Exception {
    Map<String, Integer> ma
Fold 1: category 4 examples = @Test
public void createDefaultDirectoryManagerPath() throws IOException {
    P
Fold 2: category 3 examples = @Test
public void serializeWithTruncateArrayTest() throws IOException {
    fina
Fold 2: category 4 examples = @Test
public void testToFile() throws Exception {
    item = new PathData(".", c
Fold 3: category 3 examples = @Test
public void testGenerateNewDayPairs() {
    PairCombinations pairs = getPa
Fold 3: category 4 examples = @Test
public void testEmptyByteArrayForEmptyInput() throws IOException {
    thi
Fold 4: category 3 examples = @Test
public void testToMetricResponse() throws Exception {
    String subscript
Fold 4: category 4 examples = @Test
public void testWithRevisions() {
    Country de = new Country();
    de.c


## Suitable Seed Value

In [25]:
results_by_seed = {}
for seed_try in range(1, 51):
    try:
        test_groups = create_train_test_groups(data, seed=seed_try)
    except RuntimeError:
        results_by_seed[seed_try] = "FAILED (couldn't satisfy min-per-class)"
        continue
    counts = [int((test_df["category"] == 4).sum()) for _, test_df in test_groups]
    results_by_seed[seed_try] = counts

for seed_try, res in results_by_seed.items():
    print(f"seed={seed_try}: {res}")

successes = {s: r for s, r in results_by_seed.items() if r != "FAILED (couldn't satisfy min-per-class)"}
print(f"\n{len(successes)}/{len(results_by_seed)} seeds produced a valid split.")
print("Most balanced (smallest max-fold TOD count):")
best_seed = min(successes, key=lambda s: max(successes[s]))
print(f"  seed={best_seed}: {successes[best_seed]}")

seed=1: [8, 10, 4, 71]
seed=2: [31, 5, 48, 9]
seed=3: [71, 4, 12, 6]
seed=4: [8, 9, 26, 50]
seed=5: [46, 34, 7, 6]
seed=6: [47, 9, 7, 30]
seed=7: FAILED (couldn't satisfy min-per-class)
seed=8: [34, 4, 49, 6]
seed=9: [31, 10, 7, 45]
seed=10: [11, 9, 6, 67]
seed=11: [28, 45, 13, 7]
seed=12: [7, 9, 75, 2]
seed=13: [31, 51, 7, 4]
seed=14: [9, 32, 43, 9]
seed=15: [33, 46, 9, 5]
seed=16: [48, 4, 13, 28]
seed=17: [5, 10, 7, 71]
seed=18: [31, 4, 44, 14]
seed=19: [13, 71, 9, 0]
seed=20: [10, 73, 8, 2]
seed=21: [46, 10, 4, 33]
seed=22: [73, 12, 6, 2]
seed=23: [74, 10, 9, 0]
seed=24: [36, 45, 6, 6]
seed=25: [30, 4, 51, 8]
seed=26: [31, 5, 49, 8]
seed=27: [13, 31, 5, 44]
seed=28: [4, 8, 36, 45]
seed=29: [32, 51, 7, 3]
seed=30: [13, 6, 5, 69]
seed=31: [4, 71, 15, 3]
seed=32: [50, 8, 4, 31]
seed=33: [8, 10, 66, 9]
seed=34: [31, 13, 47, 2]
seed=35: [33, 9, 46, 5]
seed=36: [7, 7, 48, 31]
seed=37: [8, 13, 68, 4]
seed=38: [5, 30, 13, 45]
seed=39: [27, 7, 7, 52]
seed=40: [50, 34, 5, 4]
seed=41: [7, 28, 